In [0]:
# ==============================================================================
# SŁOWNIKI DANYCH (LOOKUP TABLES / DIMENSIONS) - NYC TAXI TLC SPECIFICATION
# ==============================================================================
# import modules using an alias
import pyspark.sql.types as T
import pyspark.sql.functions as F
# ------------------------------------------------------------------------------
# 1. Vendor Dictionary (Dostawcy systemu taksometru)
# Klucz obcy: vendor_id
# ------------------------------------------------------------------------------
vendor_data = [
    (1, "Creative Mobile Technologies, LLC"),
    (2, "VeriFone Inc."),
    (99, "Unknown / Error")
]
# Schema: vendor_id INT, vendor_name STRING

df_vendor = (spark
             .createDataFrame(
                data=vendor_data, 
                schema=T.StructType([T.StructField("vendor_id", T.IntegerType(), False), 
                                     T.StructField("vendor_name", T.StringType(), False)])
                )
            )

# display(df_vendor)

df_vendor.write.mode("overwrite").saveAsTable("medallion_project.silver02.dim_vendor")
# ------------------------------------------------------------------------------
# 2. Rate Code Dictionary (Kody taryf i stawek)
# Klucz obcy: ratecode_id
# ------------------------------------------------------------------------------
rate_code_data = [
    (1, "Standard rate"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau or Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
    (99, "Unknown / Error")
]
# Schema: ratecode_id INT, rate_code_name STRING

df_rate_code = (spark
             .createDataFrame(
                data=rate_code_data, 
                schema=T.StructType([T.StructField("ratecode_id", T.IntegerType(), False), 
                                     T.StructField("rate_code_name", T.StringType(), False)]
             ))
)

df_rate_code.write.mode("overwrite").saveAsTable("medallion_project.silver02.dim_rate_code")

# ------------------------------------------------------------------------------
# 3. Payment Type Dictionary (Formy płatności)
# Klucz obcy: payment_type
# ------------------------------------------------------------------------------
payment_type_data = [
    (1, "Credit card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip")
]
# Schema: payment_type INT, payment_type_name STRING

df_payment_type = (spark
             .createDataFrame(
                data=payment_type_data, 
                schema=T.StructType([T.StructField("payment_type", T.IntegerType(), False), 
                                     T.StructField("payment_type_name", T.StringType(), False)]
             ))
)

df_payment_type.write.mode("overwrite").saveAsTable("medallion_project.silver02.dim_payment_type")



In [0]:
# ------------------------------------------------------------------------------
# 4. Taxi Zones / Location Dictionary (Strefy taksówkarskie)
# Klucze obce: pu_location_id, do_location_id
# Plik źródłowy: taxi_zone_lookup.csv (NYC TLC)
# ------------------------------------------------------------------------------
# Kolumny źródłowe: LocationID, Borough, Zone, service_zone
# Wartości specjalne / powszechne braki danych:
#   264 -> "Unknown", "NV" (No Zone)
#   265 -> "Unknown", "NA"
#   999 -> "Unknown / Error" (Wartość domyślna z warstwy Silver)
# Schema: location_id INT, borough STRING, zone STRING, service_zone STRING
# ==============================================================================

taxi_zone_schema = T.StructType([
    T.StructField("location_id", T.IntegerType(), True),
    T.StructField("borough", T.StringType(), True),
    T.StructField("zone", T.StringType(), True),
    T.StructField("service_zone", T.StringType(), True)
])

df_taxi_zone_lookup = (
    spark.read
    .option("header", True)
    .schema(taxi_zone_schema)  # Przekazanie gotowego schematu (zamiast inferSchema)
    .option("delimiter", ",")
    .option("encoding", "UTF-8")
    .option("nullValue", "N/A")  # Automatyczna zamiana "N/A" na NULL
    .csv("/Volumes/medallion_project/taxi_data/00_landing_data/taxi+_zone_lookup.csv")
)


df_taxi_zone_lookup.write.mode("overwrite").saveAsTable("medallion_project.silver02.dim_taxi_zone_lookup")

display(df_taxi_zone_lookup)